In [ ]:
# Install all library that needed
!pip install transformers fastapi uvicorn python-dotenv pycloudflared

In [ ]:
# Load environment and save HuggingFace token
import os
from dotenv import load_dotenv
load_dotenv("hfkey.txt")
from huggingface_hub import login
token = os.getenv("HF_TOKEN")
login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Create FastAPI application to app.py
app_py = f'''
import os
import torch
import uvicorn
from transformers import pipeline
from fastapi import FastAPI, Body

pipeline = pipeline(
    task="text-generation",
    model="budionosan/llama-finetuning-financial-customer-service",
    temperature=0.1,
    top_p=0.1
)

app = FastAPI()

@app.post("/chat")
def chat(qna: str = Body(...)):
    messages = [
        {{"role": "user", "content": qna}},
    ]

    tokenizer = pipeline.tokenizer
    response = pipeline(
        text_inputs=messages,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )
    result = response[0]['generated_text']
    return {{"input": qna, "output": result}}
'''

# Write the content to app.py
with open("app.py", "w") as f:
    f.write(app_py)

In [ ]:
import threading
import uvicorn
from pycloudflared import try_cloudflare

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()
tunnel_url = try_cloudflare(port=8000)

import time
time.sleep(5) # Waiting Cloudflare API to ready serving.

 * Running on https://webster-friend-self-prospective.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics


# **Ask about question that was related to financial customer service dataset**

In [ ]:
import requests
url = f"{tunnel_url.tunnel}/chat"

def question_answering(question):
    payload = question
    res = requests.post(url,json=payload)
    return res.json()["output"]

In [ ]:
question_answering("How do I download my checking statements from the mobile app?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To download your checking statements from the mobile app, follow these steps:\n\n1. Open the mobile app.\n2. Navigate to the Accounts section.\n3. Select your account.\n4. Choose the month.\n5. Tap the Download Statement button.'

In [ ]:
question_answering("How do I transfer funds to another savings account?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To transfer funds to another savings account, log into online banking, go to Transfers, enter the recipients account details, amount, and confirm, and transference will be processed.'

In [ ]:
question_answering("How do I set up automatic credit card payments?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To set up automatic credit card payments, follow these steps:\n\n1. Log into online banking.\n2. Go to Bill Pay.\n3. Select Auto-Pay Setup.\n4. Choose payee, amount, and schedule.\n5. Confirm the setup.'

In [ ]:
question_answering("What documents are required to open a checking account?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To open a checking account, you will need a government-issued ID, proof of address, and Social Security number. Visit our website or a branch to start the process.'

In [ ]:
question_answering("How do I dispute a charge on my credit card?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To dispute a charge on your credit card, follow these steps:\n\n1. Log into your online banking account.\n2. Navigate to the transaction you want to dispute.\n3. Select Dispute Charge.\n4. Provide the transaction details, including the amount and date.'

# **Ask about question that was similar to financial customer service dataset**

In [ ]:
question_answering("Can I access and download my checking statements via the mobile application?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'Yes, you can access and download your checking statements via the mobile application. To do this, open the app, go to Accounts, select your account, tap Statements, choose the month, and click Download PDF.'

In [ ]:
question_answering("How do I send money from my savings account to another account?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


"To send money from your savings account to another account, follow these steps:\n\n1. Log into your account.\n2. Go to Transfers.\n3. Enter the recipient's account details.\n4. Choose the amount.\n5. Confirm the transaction."

In [ ]:
question_answering("How can I enable automatic payments for my credit card?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To enable automatic payments for your credit card, follow these steps:\n\n1. Log into your online banking account.\n2. Go to Bill Pay.\n3. Select Auto-Pay Setup.\n4. Choose payee, amount, and schedule.\n5. Confirm the setup.'

In [ ]:
question_answering("Which documents are needed to open a checking account?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To open a checking account, you will need a government-issued ID, proof of address, and Social Security number. Visit our website or a branch to start the process.'

In [ ]:
question_answering("How can I dispute a credit card charge?")

INFO:     35.197.67.217:0 - "POST /chat HTTP/1.1" 200 OK


'To dispute a credit card charge, follow these steps:\n\n1. Log into your account.\n2. Navigate to the transaction you want to dispute.\n3. Select Dispute Charge.\n4. Provide the transaction details, including the amount and date.\n5. Confirm the dispute.'